In [69]:
import matplotlib
matplotlib.use('Qt5Agg')  # Important for VSCode interactivity

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import PolygonSelector
from matplotlib.path import Path
from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc

import numpy as np
import os
import plotly.io as pio
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [70]:
class PolygonAnnotator:
    def __init__(self, image, cmap='hot'):
        self.image = image
        self.coords = None
        self.done = False

        self.fig, self.ax = plt.subplots()
        self.ax.imshow(image, cmap=cmap)
        self.ax.set_title("Click to draw polygon. Double-click to finish.")

        self.selector = PolygonSelector(
            self.ax,
            self.onselect,
            useblit=True,
            props=dict(color='cyan', linewidth=2),
            handle_props=dict(marker='o', markersize=5, mec='cyan', mfc='cyan')
        )

        self.fig.canvas.mpl_connect('close_event', self._on_close)
        plt.show(block=True)  # Blocks until the window is closed

    def _on_close(self, event):
        self.done = True

    def onselect(self, verts):
        self.coords = verts
        print(f"Polygon drawn with {len(verts)} points.")
        plt.close(self.fig)  # Automatically close the window

    def get_mask(self):
        if not self.coords:
            print("No polygon was drawn.")
            return None
        ny, nx = self.image.shape
        y, x = np.mgrid[:ny, :nx]
        points = np.vstack((x.flatten(), y.flatten())).T
        path = Path(self.coords)
        return path.contains_points(points).reshape((ny, nx))

In [71]:
def get_mz_heatmap_image(adata, target_mz, tol=0.1):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    x = adata.obs["x"].values.astype(int)
    y = adata.obs["y"].values.astype(int)

    width = x.max() + 1
    height = y.max() + 1
    image = np.zeros((height, width))
    image[y, x] = intensities

    return image, matched_mz

In [72]:
# Load and visualize the m/z heatmap from AnnData
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad"
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad


In [73]:
# ---- Run ----

target_mz = 788.5
image, matched_mz = get_mz_heatmap_image(adata, target_mz)
print(f"Target m/z: {target_mz} → Matched m/z: {matched_mz}")

custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])

# Use the annotator with your image and colormap
annotator = PolygonAnnotator(image, cmap=custom_cmap)
mask = annotator.get_mask()

if mask is not None:
    masked_image = np.where(mask, image, 0)
    plt.imshow(masked_image, cmap=custom_cmap)
    plt.title(f"Masked m/z = {matched_mz:.4f}")
    plt.show()
else:
    print("No mask created.")

Target m/z: 788.5 → Matched m/z: 788.5027
Polygon drawn with 33 points.


In [74]:
# Get spatial coordinates
x = adata.obs["x"].astype(int).values
y = adata.obs["y"].astype(int).values

# Mask value at each spot
adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)
adata.obs["tissue"] = adata.obs["tissue"].astype(int)  # 1 = inside, 0 = outside
print(adata.obs["tissue"].value_counts())

tissue
0    9274
1    7331
Name: count, dtype: int64


In [75]:
adata1 = adata
adata.obs["tissue"]

0        0
1        0
2        0
3        0
4        0
        ..
16600    0
16601    0
16602    0
16603    0
16604    0
Name: tissue, Length: 16605, dtype: int64

In [130]:
adata = adata1
adata

AnnData object with n_obs × n_vars = 16605 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue'

In [131]:
def filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=2.0, layer=None, normalize_tic=True):
    # 1. Extract raw data
    data = adata.layers[layer] if layer is not None else adata.X

    # 2. TIC normalization per pixel
    if normalize_tic:
        # Sum across features for each observation (axis=1)
        tic = np.asarray(data.sum(axis=1)).ravel()
        # Avoid division by zero
        tic[tic == 0] = 1.0
        # Normalize in place (if sparse, convert to array)
        data = data / tic[:, None]

    # 3. Define tissue masks
    mask_tissue = adata.obs[tissue_key] == tissue_val
    mask_non    = adata.obs[tissue_key] == non_tissue_val

    # 4. Compute mean normalized intensity per feature
    mean_tissue    = np.asarray(data[mask_tissue].mean(axis=0)).ravel()
    mean_nontissue = np.asarray(data[mask_non].mean(axis=0)).ravel()

    # 5. Identify features to remove
    to_remove = mean_tissue < foldchange * mean_nontissue

    # 6. Print removed m/z names
    mz_to_remove = adata.var_names[to_remove]
    print(f"Dropping {len(mz_to_remove)} m/z features:", mz_to_remove.values)

    # 7. Subset and return
    return adata[:, ~to_remove].copy()

In [132]:
adata = filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=2, layer=None, normalize_tic=True)

Dropping 636 m/z features: ['273.0397' '137.0251' '274.0435' '331.0297' '257.0478' '195.0905'
 '637.3020' '313.0337' '275.0563' '245.0458' '333.0266' '197.9919'
 '330.0215' '244.0353' '246.0516' '347.0108' '551.0132' '258.0498'
 '229.0488' '501.0369' '332.0313' '544.9929' '199.0008' '375.0044'
 '290.0419' '291.0466' '638.3054' '721.0097' '367.9774' '346.0029'
 '368.9916' '409.0579' '285.0295' '543.9945' '309.1370' '550.0092'
 '262.0491' '467.0459' '483.0330' '485.0599' '307.1212' '247.0594'
 '659.2911' '138.0287' '315.0454' '353.0146' '335.0136' '523.0186'
 '272.0306' '159.0070' '297.0362' '329.0229' '269.0502' '727.0250'
 '228.0455' '256.0338' '289.0389' '314.0387' '196.0981' '276.0565'
 '361.9717' '529.0170' '352.0063' '292.0531' '271.0377' '218.0560'
 '720.0018' '219.9751' '231.0686' '287.0455' '355.0027' '566.9828'
 '548.9960' '259.0608' '334.0233' '449.0490' '705.0255' '295.0229'
 '255.0287' '528.0132' '552.0181' '897.0110' '288.0302' '350.9995'
 '903.0234' '500.0308' '502.0343' '

In [133]:
adata

AnnData object with n_obs × n_vars = 16605 × 364
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue'

In [134]:
adata.var_names

Index(['369.3523', '798.5391', '760.5916', '772.5231', '734.5673', '799.5511',
       '370.3516', '782.5675', '761.5914', '184.0720',
       ...
       '1542.1475', '1544.1542', '602.0143', '215.1771', '702.5467',
       '252.9273', '386.2827', '621.6046', '703.5655', '564.2518'],
      dtype='object', length=364)

In [135]:
def plot_mz_across_sample_custom_tic(adata, target_mz, tol=0.1):
    """
    Plots the TIC-normalized intensity of a specific m/z across all pixels using spatial coordinates.

    Parameters:
    - adata: AnnData object with MSI data
    - target_mz: float, the m/z value of interest
    - tol: float, tolerance for matching m/z (default ±0.1)
    """
    # Convert var_names to float (m/z axis)
    mz_axis = adata.var_names.astype(float).values

    # Find index of the m/z closest to the target within tolerance
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    # Extract intensities for that m/z from all pixels
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]

    # Normalize by TIC
    if "TIC" not in adata.obs.columns:
        raise KeyError("TIC not found in adata.obs. Compute TIC before using this function.")
    tic = adata.obs["TIC"].values
    norm_intensities = intensities / tic
    norm_intensities[np.isnan(norm_intensities)] = 0  # Avoid NaNs from division by 0

    # Extract spatial coordinates
    if "x" not in adata.obs.columns or "y" not in adata.obs.columns:
        raise KeyError("Spatial coordinates 'x' and 'y' not found in adata.obs")

    x = adata.obs["x"].values
    y = adata.obs["y"].values

    # Create DataFrame for plotting
    df = pd.DataFrame({"x": x, "y": y, "intensity": norm_intensities})

    # Plot as heatmap using plotly express
    fig = px.scatter(
        df,
        x="x",
        y="y",
        color="intensity",
        color_continuous_scale=[
            [0.0, "#000000"],  # black
            [0.10, "#000080"],  # navy
            [0.15, "#0000FF"],  # blue
            [0.30, "#8000FF"],  # purple-ish
            [0.45, "#FF0000"],  # red
            [0.60, "#FF8000"],  # orange
            [0.75, "#FFFF00"],  # yellow
            [1.0, "#FFFFFF"]   # white
        ],
        title=f"TIC-Normalized Intensity Map of m/z = {matched_mz:.4f}",
        labels={"intensity": "Normalized Intensity"},
        height=500
    )

    fig.update_layout(yaxis=dict(scaleanchor="x", autorange="reversed"))
    return fig

In [136]:


# Ensure the image folder exists
os.makedirs("images", exist_ok=True)

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices from 10th to 20th (ranked #10 to #20)
selected_indices = sorted_indices[0:]

# Get corresponding m/z values
selected_mz = adata.var_names[selected_indices].astype(float)

# Modified plotting loop
for mz in selected_mz:
    # Generate figure using your existing function
    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Modify function to return fig
    
    # Define filename
    filename = f"images_9w_3p_3points_tic/mz_{mz:.4f}.png"
    
    # Save the figure
    pio.write_image(fig, filename)
    print(f"Saved: {filename}")

Saved: images_9w_3p_3points_tic/mz_369.3523.png
Saved: images_9w_3p_3points_tic/mz_734.5673.png
Saved: images_9w_3p_3points_tic/mz_798.5391.png
Saved: images_9w_3p_3points_tic/mz_370.3516.png
Saved: images_9w_3p_3points_tic/mz_184.0720.png
Saved: images_9w_3p_3points_tic/mz_760.5916.png
Saved: images_9w_3p_3points_tic/mz_772.5231.png
Saved: images_9w_3p_3points_tic/mz_735.5735.png
Saved: images_9w_3p_3points_tic/mz_367.3328.png
Saved: images_9w_3p_3points_tic/mz_651.5335.png
Saved: images_9w_3p_3points_tic/mz_348.0689.png
Saved: images_9w_3p_3points_tic/mz_799.5511.png
Saved: images_9w_3p_3points_tic/mz_782.5675.png
Saved: images_9w_3p_3points_tic/mz_756.5510.png
Saved: images_9w_3p_3points_tic/mz_230.9492.png
Saved: images_9w_3p_3points_tic/mz_761.5914.png
Saved: images_9w_3p_3points_tic/mz_731.6000.png
Saved: images_9w_3p_3points_tic/mz_627.5297.png
Saved: images_9w_3p_3points_tic/mz_368.3461.png
Saved: images_9w_3p_3points_tic/mz_366.9731.png
Saved: images_9w_3p_3points_tic/mz_800.5